In [2]:
import pandas as pd
from bs4 import BeautifulSoup

In [3]:
soup = BeautifulSoup(open("../../../data/data/drugbank/Drugbank_Fulldata.xml"),"xml")

In [8]:
drugs = soup.findAll("drug")

/tmp/ipykernel_2172409/522772564.py:1: DeprecationWarning: Call to deprecated method findAll. (Replaced by find_all) -- Deprecated since version 4.0.0.
  drugs = soup.findAll("drug")


In [9]:
data = []

for idx, drug in enumerate(drugs):
    print(f'{idx+1}/{len(drugs)}', end='\r')
    if len(drug.find_all(recursive=False)) <= 2:
        continue
    id_ = drug.find("drugbank-id").text
    name = drug.find("name").text
    
    data.append({
        'id': id_,
        'name': name,
    })
    
    general_infomation = [
        'volume-of-distribution',  # 分布容积
        'half-life',  # 半衰期
        'pharmacodynamics',  # 药效学
        'description',  # 描述
        'simple-description',  # 简单描述
        'clinical-description',  # 临床描述
        'state',  # 状态：固体/液体/气体
        'indication',  # 适应症
        'protein-binding',  # 蛋白质结合
        'mechanism-of-action',  # 作用机制
        'toxicity',  # 毒性
        'metabolism',  # 代谢
        'absorption',  # 吸收
        'route-of-elimination', # 消除途径
        'clearance',  # 清除率
    ]
    
    for v in general_infomation:
        if (ele:= drug.find(v)) is not None:
            data[-1][v] = ele.text.strip()
         
    sequences = [sequence.text.strip() for sequence in drug.find_all('sequence')]  # 可能包含多条肽链
    if len(sequences) > 0:   
        data[-1]['sequences'] = '|'.join(sequences)
    
    for prop in drug.find_all('property'):  # SMILES
        kind = prop.find('kind').text.strip()      
        if kind == 'SMILES':
            data[-1]['SMILES'] = prop.find('value').text.strip()
            break

In [10]:
df = pd.DataFrame(data)
df.to_csv('../../../data/data_feature/drugbank.csv', index=False)

In [11]:
df.head()

,id,name,volume-of-distribution,half-life,pharmacodynamics,description,state,indication,protein-binding,mechanism-of-action,toxicity,metabolism,absorption,route-of-elimination,clearance,sequences,SMILES
0,DB00001,Lepirudin,The volume of distribution of lepirudin at ste...,Lepirudin has an initial half-life of approxim...,Lepirudin is a recombinant hirudin that acts a...,Lepirudin is a recombinant hirudin formed by 6...,solid,Lepirudin is indicated for anticoagulation in ...,"In human plasma, the protein binding of lepiru...",Lepirudin is a direct thrombin inhibitor used ...,The acute toxicity of intravenous lepirudin wa...,"As a polypeptide, lepirudin is expected to be ...",Lepirudin administered as a single intravenous...,Lepirudin is mostly excreted through urine (48...,The clearance of lepirudin is proportional to ...,>DB00001 sequence\nLTYTDCTESGQNLCLCEGSNVCGQGNK...,NaN
1,DB00002,Cetuximab,The volume of the distribution is about 2-3 L/...,After administration of a 400 mg/m<sup>2</sup>...,Cetuximab is an anticancer agent that works by...,Cetuximab is a recombinant chimeric human/mous...,liquid,Cetuximab indicated for the treatment of local...,There is no information available.,The epidermal growth factor receptor (EGFR) is...,The intravenous LD<sub>50</sub> is > 300 mg/kg...,"Like other monoclonal antibodies, cetuximab is...",After administration of a 400 mg/m<sup>2</sup>...,There is limited information available.,In patients with recurrent and/or metastatic s...,>Cetuximab heavy chain\nQVQLKQSGPGLVQPSQSLSITC...,NaN
2,DB00003,Dornase alfa,"In studies in rats and monkeys, the initial vo...",,Cystic fibrosis (CF) is a disease characterize...,Dornase alfa is a biosynthetic form of human d...,liquid,Used as adjunct therapy in the treatment of cy...,,Dornase alfa is a biosynthetic form of human D...,Adverse reactions occur at a frequency of < 1/...,While no conclusive studies have yet been publ...,Studies in rats and monkeys after inhalation o...,,"Studies in rats indicate that, following aeros...",>Dornase alfa sequence\nLKIAAFNIQTFGETKMSNATLV...,NaN
3,DB00004,Denileukin diftitox,The geometric mean (CV%) volume of distributio...,The arithmetic mean (CV%) denileukin diftitox ...,Denileukin diftitox is an anticancer drug with...,Denileukin diftitox is an IL2-receptor-directe...,liquid,Denileukin diftitox was previously indicated f...,,Denileukin diftitox is a fusion protein compos...,There is limited information regarding the acu...,Denileukin diftitox is expected to be metaboli...,Following a single dose of denileukin diftitox...,,The geometric mean (CV%) clearance is 36.5 mL/...,NaN,NaN
4,DB00005,Etanercept,Population pharmacokinetic modeling predicts a...,Etanercept has a mean half-life of elimination...,Etanercept binds specifically to tumor necrosi...,Dimeric fusion protein consisting of the extra...,liquid,Etanercept is indicated for the treatment of m...,No significant protein binding has been identi...,There are two distinct receptors for TNF (TNFR...,,"As etanercept is a fusion protein antibody, it...",Population pharmacokinetic modeling in adults ...,,Etanercept has a mean apparent clearance of 16...,> Etanercept Sequence\nLPAQVAFTPYAPEPGSTCRLREY...,NaN
